 Fine-tune only small LoRA modules for learning, saving time and resources while still achieving good performance in classifying emotions such as joy, sadness, anger, love, fear and surprise.

In [ ]:
!pip install transformers datasets peft accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.9 MB/s eta 0:00:00


In [ ]:
# Import necessary classes from the transformers library for sequence classification.
# AutoModelForSequenceClassification loads a pre-trained model for classification tasks.
# AutoTokenizer loads the corresponding tokenizer for the model.
# TrainingArguments defines the configuration for the Trainer.
# Trainer is a high-level API for training models.
# pipeline provides a simple way to use models for inference.
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TrainingArguments, Trainer, pipeline
# Import load_dataset from the datasets library to easily load and manage datasets.
from datasets import load_dataset
# Import LoraConfig for configuring LoRA (Low-Rank Adaptation) and get_peft_model for applying PEFT to a base model.
from peft import LoraConfig, get_peft_model
# Import the evaluate library to compute various metrics during evaluation.
import evaluate
# Import the torch library, a popular open-source machine learning framework.
import torch

In [ ]:
# Load the 'dair-ai/emotion' dataset from the Hugging Face Hub.
dataset = load_dataset("dair-ai/emotion")
# Shuffle the training split for randomness and select a subset of 3000 examples with a fixed seed for reproducibility.
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(3000))
# Shuffle the validation split and select a subset of 500 examples.
dataset["validation"] = dataset["validation"].shuffle(seed=42).select(range(500))
# Shuffle the test split and select a subset of 500 examples.
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(500))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [ ]:
# This section is dedicated to preprocessing the data for model training.
# Define the name of the pre-trained model to be used, in this case, BERT base uncased.
model_name = "bert-base-uncased"
# Initialize a tokenizer using the pre-trained model to convert text into numerical input IDs and attention masks.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Define a function to preprocess each batch of the dataset.
def preprocess(batch):
  # Tokenize the text in the batch.
  # truncation=True ensures that longer sequences are cut to max_length.
  # padding="max_length" pads shorter sequences to max_length.
  # max_length=128 sets the maximum sequence length for tokenization.
  return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)
# Apply the preprocessing function to the entire dataset in batches.
# rename_column("label", "labels") renames the 'label' column to 'labels', which is expected by Hugging Face transformers models.
dataset = dataset.map(preprocess, batched=True).rename_column("label", "labels")
# Set the format of the dataset to PyTorch tensors, specifying the required columns.
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
# Load the pre-trained AutoModelForSequenceClassification model.
# num_labels=6 specifies that the model should have 6 output classes for emotion classification.
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=6)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# This section defines the configuration for LoRA (Low-Rank Adaptation), a PEFT technique.
# Initialize the LoraConfig object with specific parameters.
lora_config = LoraConfig(
    # r is the rank of the update matrices. A smaller r means fewer trainable parameters.
    r=8,
    # lora_alpha is a scaling factor for the LoRA updates.
    lora_alpha=16,
    # target_modules specifies which layers of the model LoRA should be applied to (e.g., attention query and value matrices).
    target_modules=["query","value"],
    # lora_dropout applies dropout to the LoRA layers to prevent overfitting.
    lora_dropout=0.1,
    # bias="none" means no bias terms are added to the LoRA layers.
    bias="none",
    # task_type specifies the type of task, in this case, Sequence Classification.
    task_type="SEQ_CLS"
)

# Apply the PEFT (LoRA) configuration to the base model.
# This modifies the model to include LoRA layers, making it a PEFT-enabled model.
model = get_peft_model(model, lora_config)

In [ ]:
# This section sets up and executes the training loop for the LoRA model.

# Load the "accuracy" metric from the evaluate library to track performance.
metric = evaluate.load("accuracy")

# Define a function to compute custom metrics during evaluation.
def compute_metrics(eval_pred):
  # Unpack the evaluation predictions into raw logits and true labels.
  logits, labels = eval_pred
  # Get the predicted class by finding the index of the maximum logit for each example.
  preds = logits.argmax(axis=-1)
  # Compute and return the accuracy using the predicted and true labels.
  return metric.compute(predictions=preds, references=labels)

# Initialize TrainingArguments to configure the training process.
lora_args = TrainingArguments(
    # output_dir specifies the directory where model checkpoints and training outputs will be saved.
    output_dir="./lora_emotion",
    # per_device_train_batch_size sets the batch size for training on each GPU/CPU.
    per_device_train_batch_size=8,
    # per_device_eval_batch_size sets the batch size for evaluation on each GPU/CPU.
    per_device_eval_batch_size=8,
    # learning_rate sets the initial learning rate for the optimizer.
    learning_rate=2e-4,
    # num_train_epochs specifies the total number of training epochs.
    num_train_epochs=5,
    # eval_strategy="epoch" means evaluation will be performed at the end of each epoch.
    eval_strategy="epoch",
    # logging_steps sets how often (in steps) training information is logged.
    logging_steps=10,
    # report_to="none" disables reporting metrics to external services like Weights & Biases.
    report_to="none"
)

# Initialize the Trainer object, which orchestrates the training and evaluation loop.
trainer = Trainer(
    # model is the PEFT-enabled model to be trained.
    model=model,
    # args are the TrainingArguments defined above.
    args=lora_args,
    # train_dataset is the dataset used for training.
    train_dataset= dataset["train"],
    # eval_dataset is the dataset used for evaluation (validation).
    eval_dataset=dataset["validation"],
    # compute_metrics provides the function to calculate custom metrics during evaluation.
    compute_metrics=compute_metrics
)

# Start the training process.
trainer.train()
# Save only the trained LoRA adapter weights to the specified directory.
model.save_pretrained("./lora_emotion_adapter")
# Print a confirmation message indicating that fine-tuning is complete and the adapter is saved.
print("LoRA Fine-tuning Complete! Adapter saved at ./lora_emotion_adapter")

Epoch,Training Loss,Validation Loss,Accuracy
1,0.875079,0.877391,0.682000
2,0.586177,0.744651,0.750000
3,0.659315,0.700764,0.760000
4,0.436649,0.607937,0.796000
5,0.408184,0.608360,0.790000


LpRA Fine-tuning Complete! Adapter saveda at ./lora_emotion_adapter


In [ ]:
# This section demonstrates how to use the fine-tuned LoRA model for inference.

# Determine the device to run the model on (GPU if available, otherwise CPU).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Move the model to the selected device.
model.to(device)

# Get the human-readable names for the emotion labels from the training dataset features.
emotion_labels = dataset["train"].features["labels"].names

# Define a list of sample text inputs for which we want to predict emotions.
samples = [
    "I am so happy to see you!",
    "This is terrifying, I can't handle it.",
    "He surprised everyone with his gift.",
    "I feel so sad and lonely.",
    "I love spending time with my family."
]

# Tokenize the sample texts.
# truncation=True truncates sequences longer than the model's maximum input length.
# padding=True pads shorter sequences to the maximum length of the batch.
# return_tensors="pt" returns the output as PyTorch tensors.
# .to(device) moves the tokenized inputs to the selected device.
inputs = tokenizer(samples, truncation=True, padding=True, return_tensors="pt").to(device)

# Set the model to evaluation mode.
# This disables dropout and other training-specific layers.
model.eval()

# Disable gradient calculations during inference to save memory and speed up computation.
with torch.no_grad():
  # Pass the tokenized inputs through the model to get the raw output logits.
  logits = model(**inputs).logits
  # Get the predicted class by finding the index of the maximum logit for each sample.
  preds = torch.argmax(logits, dim=-1)

# Iterate through the original sample texts and their corresponding predictions.
for text, pred in zip(samples, preds):
  # Print the original text and its predicted emotion using the human-readable label names.
  print(f"Text: {text}\nPredicted Emotion: {emotion_labels[pred]}\n")

Text: I am so happy to see you!
Predicted Emotion: joy

Text: This is terrifying, I can't handle it.
Predicted Emotion: fear

Text: He surprised everyone with his gift.
Predicted Emotion: joy

Text: I feel so sad and lonely.
Predicted Emotion: sadness

Text: I love spending time with my family.
Predicted Emotion: love

